# Diabetes Classification & ROC-AUC Evaluation

A machine-learning practice project focused on binary classification and model evaluation, especially ROC-AUC.

**Dataset:** `diabetes.csv`  
**Target:** `glyhb` converted to a binary diabetes label using a 6.5 threshold.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, RocCurveDisplay
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

## Part 2 – Initial Data Exploration

In [ ]:
# Load dataset
df = pd.read_csv('diabetes.csv')
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nInfo:")
df.info()

print("\nSummary statistics:")
display(df.describe())

### Observations
- 403 rows, 20 columns.
- Missing values in many columns (glyhb, bp.2s, bp.2d, etc.).
- Target `glyhb` ranges from 2.68 to 16.11; threshold 6.5 gives ~26% diabetic (imbalanced).
- `id` is an identifier, irrelevant for prediction.
- Categorical features: location, gender, frame.

## Part 3 – Data Quality Assessment and Cleaning

In [ ]:
# 1. Missing values
missing_pct = df.isnull().mean() * 100
print("Missing percentages:")
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

# Treatment:
df_clean = df.dropna(subset=['glyhb']).copy()
df_clean.drop(['bp.2s', 'bp.2d'], axis=1, inplace=True)   # too many missing

num_cols = ['chol', 'stab.glu', 'hdl', 'ratio', 'age', 'height', 'weight', 
            'bp.1s', 'bp.1d', 'waist', 'hip', 'time.ppn']
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

cat_cols = ['location', 'gender', 'frame']
for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f"\nRemaining missing after cleaning: {df_clean.isnull().sum().sum()}")

# 2. Duplicates
print(f"Duplicate rows: {df_clean.duplicated().sum()}")

# 3. Outliers (IQR)
def count_outliers(df, cols, threshold=1.5):
    res = {}
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - threshold*iqr
        upper = q3 + threshold*iqr
        res[col] = len(df[(df[col] < lower) | (df[col] > upper)])
    return res

out = count_outliers(df_clean, num_cols)
print("\nOutliers per feature:")
for k,v in out.items():
    print(f"{k}: {v}")

# 4. Create target
df_clean['diabetes'] = (df_clean['glyhb'] > 6.5).astype(int)
df_clean.drop(['glyhb', 'id'], axis=1, inplace=True)

print("\nTarget distribution:")
print(df_clean['diabetes'].value_counts(normalize=True))

## Part 4 – Exploratory Data Analysis (EDA)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14,10))
sns.heatmap(df_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Box plots by target
features = ['chol', 'stab.glu', 'hdl', 'ratio', 'age', 'weight', 'waist', 'hip']
fig, axes = plt.subplots(2, 4, figsize=(16,8))
for i, feat in enumerate(features):
    sns.boxplot(x='diabetes', y=feat, data=df_clean, ax=axes[i//4, i%4])
plt.tight_layout()
plt.show()

# Categorical vs target
print("Gender vs diabetes:")
print(pd.crosstab(df_clean['gender'], df_clean['diabetes'], normalize='index'))
print("\nLocation vs diabetes:")
print(pd.crosstab(df_clean['location'], df_clean['diabetes'], normalize='index'))
print("\nFrame vs diabetes:")
print(pd.crosstab(df_clean['frame'], df_clean['diabetes'], normalize='index'))

# EDA Insights (written in markdown after this cell)

### EDA Insights
1. Diabetic patients have higher mean age, weight, waist, hip, and stab.glu.
2. HDL (good cholesterol) is lower in diabetics.
3. Ratio (chol/hdl) is higher in diabetics (dyslipidemia).
4. Age is a key predictor; diabetes risk increases with age.
5. Large frame and Buckingham location show slightly higher diabetes rates.
6. Waist and hip (central obesity) strongly correlate with diabetes.

## Part 5 – Feature Preparation

In [ ]:
X = df_clean.drop('diabetes', axis=1)
y = df_clean['diabetes']

X_encoded = pd.get_dummies(X, columns=['location', 'gender', 'frame'], drop_first=True)

num_features = ['chol', 'stab.glu', 'hdl', 'ratio', 'age', 'height',
                'weight', 'bp.1s', 'bp.1d', 'waist', 'hip', 'time.ppn']

print("Encoded feature matrix shape:", X_encoded.shape)

## Part 6 – Data Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaling parameters on training data only.
scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(y_train.value_counts(normalize=True))

## Part 7 – Baseline Model

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=42)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print("Baseline Logistic Regression (Test):")
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_base):.4f}")
print(f"F1: {f1_score(y_test, y_pred_base):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_base))

## Part 8 – Model Development (Multiple Algorithms)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    cv_scores[name] = scores.mean()
    print(f"{name}: CV F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

## Part 9 – Model Evaluation (Test Set)

In [ ]:
test_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    }

pd.DataFrame(test_results).T

## Part 10 – Overfitting / Underfitting Analysis

In [ ]:
train_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    train_scores[name] = f1_score(y_train, y_train_pred)

comparison = pd.DataFrame({
    'Train F1': train_scores,
    'CV F1': cv_scores,
    'Test F1': [test_results[n]['F1'] for n in models.keys()]
})
comparison['Train-CV Gap'] = comparison['Train F1'] - comparison['CV F1']
display(comparison)

# Interpretation:
# - Decision Tree: large gap (>0.3) → overfitting.
# - Random Forest & Gradient Boosting: small gap → good generalization.
# - Logistic Regression: small gap, slightly underfit but stable.

## Part 11 – Hyperparameter Experiments

In [ ]:
# Random Forest tuning
param_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_rf, cv=5, scoring='f1')
grid_rf.fit(X_train, y_train)
print("Best RF params:", grid_rf.best_params_)
print("Best CV F1:", grid_rf.best_score_)

# Gradient Boosting tuning
param_gb = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_gb, cv=5, scoring='f1')
grid_gb.fit(X_train, y_train)
print("\nBest GB params:", grid_gb.best_params_)
print("Best CV F1:", grid_gb.best_score_)

## Part 12 – Model Comparison and Final Selection

In [ ]:
final_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest (tuned)': RandomForestClassifier(**grid_rf.best_params_, random_state=42),
    'Gradient Boosting (tuned)': GradientBoostingClassifier(**grid_gb.best_params_, random_state=42)
}

final_results = {}
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    }

final_df = pd.DataFrame(final_results).T
display(final_df)

# Final model selection: Gradient Boosting (highest F1, good generalization, interpretable feature importance).
gb_final = final_models['Gradient Boosting (tuned)']

# Feature importance
importances = gb_final.feature_importances_
features = X_train.columns
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10,6))
plt.title('Gradient Boosting Feature Importance')
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), [features[i] for i in indices], rotation=90)
plt.tight_layout()
plt.show()

## Part 13 – Error Analysis

In [ ]:
y_pred_gb = gb_final.predict(X_test)
cm = confusion_matrix(y_test, y_pred_gb)
print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"\nFalse Negatives (missed diabetics): {fn}")
print(f"False Positives (false alarms): {fp}")

# Examine false negatives indices
fn_idx = np.where((y_test == 1) & (y_pred_gb == 0))[0]
print(f"\nSample of false negatives (scaled features):")
print(X_test.iloc[fn_idx].head())

# Error insights (written in markdown after)

### Error Analysis Findings
1. False Negatives (missed cases) are more critical than False Positives because they leave diabetics undiagnosed.
2. Model may misclassify borderline cases (glyhb near 6.5).
3. Some false negatives have high waist/hip and age, suggesting the model might benefit from a lower decision threshold or cost-sensitive learning.

## ROC-AUC Analysis

Precision, recall, and F1 describe performance at a chosen threshold. ROC-AUC summarizes how well a model ranks positive cases above negative cases across thresholds and is calculated from predicted probabilities.

In [ ]:
roc_auc_results = {}
for name, model in final_models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    roc_auc_results[name] = roc_auc_score(y_test, y_prob)

display(pd.Series(roc_auc_results, name='ROC-AUC').sort_values(ascending=False).to_frame())

RocCurveDisplay.from_estimator(gb_final, X_test, y_test)
plt.title('ROC Curve — Gradient Boosting')
plt.show()

## Part 14 – Unsupervised Learning (Clustering)

In [ ]:
X_cluster = X_encoded.copy()
X_cluster[num_features] = StandardScaler().fit_transform(X_cluster[num_features])

# K-Means: elbow & silhouette
inertia = []
sil = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_cluster, km.labels_))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow')
plt.subplot(1,2,2)
plt.plot(K_range, sil, marker='o')
plt.title('Silhouette')
plt.show()

# Best K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)
df_cluster = X_cluster.copy()
df_cluster['cluster'] = cluster_labels
df_cluster['diabetes'] = y.values

# Profile
print("Cluster profiles (mean of scaled features):")
display(df_cluster.groupby('cluster').mean())

print("\nDiabetes rate per cluster:")
print(pd.crosstab(df_cluster['cluster'], df_cluster['diabetes'], normalize='index'))

# Hierarchical clustering
hier = AgglomerativeClustering(n_clusters=3)
hier_labels = hier.fit_predict(X_cluster)
print(f"\nHierarchical Silhouette: {silhouette_score(X_cluster, hier_labels):.3f}")

# Interpretation: Cluster 1 has the highest diabetes proportion, representing high-risk group.

## Part 15 – Dimensionality Reduction (PCA)

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_cluster)
cum_var = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8,4))
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o')
plt.axhline(0.95, color='r', linestyle='--', label='95%')
plt.xlabel('Components')
plt.ylabel('Cumulative variance')
plt.title('PCA Explained Variance')
plt.legend()
plt.show()

n_comp = np.argmax(cum_var >= 0.95) + 1
print(f"Retain {n_comp} components to explain 95% variance.")

pca = PCA(n_components=n_comp)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

gb_pca = GradientBoostingClassifier(**grid_gb.best_params_, random_state=42)
gb_pca.fit(X_train_pca, y_train)
y_pred_pca = gb_pca.predict(X_test_pca)
f1_pca = f1_score(y_test, y_pred_pca)
print(f"Gradient Boosting with PCA F1: {f1_pca:.4f}")
print(f"Original (without PCA) F1: {final_results['Gradient Boosting (tuned)']['F1']:.4f}")

# Conclusion: PCA simplifies features but slightly reduces performance; not crucial for this dataset.